# Day 2 — RAG: Chunking, Embeddings & Vector Store

This notebook builds the retrieval/indexing side of a RAG system: loading documents → chunking → embedding chunks → storing them in Chroma → inspecting and visualizing the resulting vectors.

## Concept

This notebook focuses on the indexing side of RAG. Before a user asks a question, the knowledge base is prepared so that relevant information can later be retrieved efficiently.

In [1]:
pip install tiktoken numpy python-dotenv langchain-openai langchain-chroma langchain-huggingface langchain-community langchain-text-splitters sentence-transformers scikit-learn plotly

  Using cached tiktoken-0.13.0-cp314-cp314-macosx_11_0_arm64.whl.metadata (6.7 kB)
  Using cached numpy-2.5.2-cp314-cp314-macosx_14_0_arm64.whl.metadata (6.6 kB)
  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached langchain_openai-1.4.3-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_chroma-1.1.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached langchain_huggingface-1.2.2-py3-none-any.whl.metadata (4.0 kB)
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_text_splitters-1.1.2-py3-none-any.whl.metadata (3.3 kB)
  Using cached sentence_transformers-5.7.0-py3-none-any.whl.metadata (18 kB)
  Using cached scikit_learn-1.9.0-cp314-cp314-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached plotly-6.9.0-py3-none-any.whl.metadata (9.0 kB)
  Using cached regex-2026.7.19-cp314-cp314-macosx_11_0_arm64.whl.metadata (40 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached 

### PART A: Divide our documents into chunks

**Concept:** These libraries handle different parts of the RAG pipeline. `DirectoryLoader` and `TextLoader` load the knowledge-base documents, `RecursiveCharacterTextSplitter` divides them into chunks, `HuggingFaceEmbeddings` converts chunks into vectors, `Chroma` stores those vectors, and `TSNE`/Plotly are used to visualize the vectors. The notebook also imports `tiktoken` to count tokens.

In [3]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go

/Users/nasir/Learnings/llm-engg-self/llm-engineering/RAG/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/77/h4yvwb4n3_387bc896pz1lk00000gn/T/ipykernel_7119/3216454956.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


**Concept:** The notebook chooses `gpt-4.1-nano` as its low-cost model because the fictional company prioritizes cost. `db_name` specifies where the Chroma vector database will be stored. `load_dotenv()` loads environment variables such as the API key from `.env`. The `MODEL` here is used for token counting; the actual chunk embeddings later are created using a separate Hugging Face embedding model.

In [4]:
# price is a factor for our company, so we're going to use a low cost model

MODEL = "gpt-4.1-nano"
db_name = "vector_db"
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

OpenAI API Key not set


**Concept:** This searches the `knowledge-base` directory for all Markdown files and combines their contents into one string. The notebook first measures the total amount of text so we can understand how large the knowledge base is before chunking it.

In [5]:
# How many characters in all the documents?

knowledge_base_path = "knowledge-base/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base")

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")

Found 76 files in the knowledge base
Total characters in knowledge base: 304,434


**Concept:** Characters and tokens are different measurements. Since LLMs process tokens rather than characters, token count gives us a more meaningful estimate of how large the knowledge base is from an LLM's perspective. `tiktoken` tokenizes the entire knowledge base using the encoding associated with the selected model and counts the resulting token IDs.

In [ ]:
# How many tokens in all the documents?

encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode(entire_knowledge_base)
token_count = len(tokens)
print(f"Total tokens for {MODEL}: {token_count:,}")

**Concept:** Instead of manually reading files into one large string, LangChain's document loaders create structured `Document` objects. Each document contains its text plus metadata. The notebook also adds a `doc_type` such as `products`, `employees`, `contracts`, or `company`, which will later help identify and visualize where each chunk came from.

In [6]:
# Load in everything in the knowledgebase using LangChain's loaders

folders = glob.glob("knowledge-base/*")

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

Loaded 76 documents


**Concept:** This simply lets you inspect one of the loaded LangChain `Document` objects. You can see both its textual content and its metadata.

In [7]:
documents[1]

Document(metadata={'source': 'knowledge-base/products/Claimllm.md', 'doc_type': 'products'}, page_content="# Product Summary\n\n# Claimllm\n\n## Summary\n\nClaimllm is Insurellm's revolutionary claims processing platform that transforms the claims experience for insurers, adjusters, and policyholders. Powered by advanced AI, machine learning, and computer vision, Claimllm automates claims handling across all insurance lines—from first notice of loss through final settlement. By dramatically reducing processing time, improving accuracy, and enhancing fraud detection, Claimllm enables insurers to deliver exceptional claims service while significantly reducing operational costs. The platform seamlessly integrates with existing policy administration and core systems to create a unified insurance ecosystem.\n\n## Features\n\n### 1. Intelligent FNOL Processing\nClaimllm's AI-powered first notice of loss intake captures claim details through multiple channels including mobile apps, web portal

**Concept:** RAG generally does not embed an entire large document as one vector. Instead, documents are divided into smaller chunks, because retrieval needs to identify the specific section relevant to a user's question. Here, each chunk has a target size of 1000 characters, and consecutive chunks overlap by 200 characters. The overlap helps preserve context when an important piece of information lies near a chunk boundary.

```text
Document
   ↓
Chunking
   ↓
Chunk 1
Chunk 2
Chunk 3
...
   ↓
Embedding
   ↓
One vector per chunk
```

In [8]:
# Divide into chunks using the RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 413 chunks
First chunk:

page_content='# Product Summary

# Rellm: AI-Powered Enterprise Reinsurance Solution

## Summary

Rellm is an innovative enterprise reinsurance product developed by Insurellm, designed to transform the way reinsurance companies operate. Harnessing the power of artificial intelligence, Rellm offers an advanced platform that redefines risk management, enhances decision-making processes, and optimizes operational efficiencies within the reinsurance industry. With seamless integrations and robust analytics, Rellm enables insurers to proactively manage their portfolios and respond to market dynamics with agility.

## Features

### AI-Driven Analytics
Rellm utilizes cutting-edge AI algorithms to provide predictive insights into risk exposures, enabling users to forecast trends and make informed decisions. Its real-time data analysis empowers reinsurance professionals with actionable intelligence.' metadata={'source': 'knowledge-base/products/Rellm.md', '

documents = list of documents; split_documents() = go inside each document and split its text; chunks = list of smaller Document objects.

**Concept:** This is simply another inspection step so you can see what the resulting chunks actually look like. Chunking quality has a direct impact on RAG retrieval quality. (basically, inspecting one of the chunks here)

In [9]:
chunks[100]

Document(metadata={'source': 'knowledge-base/contracts/Contract with National Claims Network for Claimllm.md', 'doc_type': 'contracts'}, page_content="7. **Business Continuity:** Insurellm provides disaster recovery with 4-hour RTO (Recovery Time Objective) and 1-hour RPO (Recovery Point Objective).\n\n---\n\n## Renewal\n\nThis agreement includes a mutual 120-day renewal notice period. National Claims Network receives guaranteed enterprise pricing for renewal equal to or better than new enterprise customers at renewal time. Contract may be extended in 12-month increments with mutual written agreement.\n\n---\n\n## Features\n\nNational Claims Network will receive the complete Claimllm Enterprise suite:\n\n1. **Unlimited Claims Processing:** No volume restrictions, supporting National's processing of 100,000+ claims annually with scalability to 500,000+ claims as business grows.\n\n2. **White-Label Platform:** Complete branding customization including:\n   - Custom domain names (claims.n

### PART B: Make vectors and store in Chroma

In Week 3, you set up a Hugging Face account and got an HF_TOKEN

At this point, you might want to add it to your `.env` file and run `load_dotenv(override=True)`

(This actually shouldn't be required).

**Concept:** This is one of the most important cells. An embedding model converts each text chunk into a numerical vector representing its semantic meaning. Here the notebook uses the open-source Hugging Face model `all-MiniLM-L6-v2`. The commented-out alternative shows that an OpenAI embedding model could also be used. The embedding model is separate from the generative LLM.

```text
Chunk
  ↓
Embedding Model
  ↓
[0.21, -0.53, 0.17, ...]
  ↓
Chroma
```

In [10]:
# Pick an embedding model

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
#embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 15878.17it/s]


Vectorstore created with 413 documents


**Concept:** This checks what was actually stored in Chroma. `count` tells us how many embedded chunks exist, while `dimensions` tells us how many numbers make up each embedding vector.

```text
1 chunk → 1 embedding vector
```

In [11]:
# Let's investigate the vectors

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 413 vectors with 384 dimensions in the vector store


### Part C: Visualize!

**Concept:** The notebook retrieves the vectors, original document chunks, and metadata from Chroma so they can be visualized. Each vector exists in a high-dimensional space, while `doc_type` tells us what category the corresponding chunk belongs to.

In [12]:
# Prework

result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['doc_type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

**Concept:** Embeddings may have hundreds or thousands of dimensions, which humans cannot directly visualize. t-SNE (t-distributed Stochastic Neighbor Embedding) reduces these high-dimensional vectors to 2 dimensions while attempting to preserve local relationships between nearby points. This is only for visualization; the actual vectors in the database remain high-dimensional.

```text
Original: [768 dimensions]
        ↓
       t-SNE
        ↓
     [2 dimensions]
        ↓
       Plot
```

In [13]:
# We humans find it easier to visalize things in 2D!
# Reduce the dimensionality of the vectors to 2D using t-SNE
# (t-distributed stochastic neighbor embedding)

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

**Concept:** This creates the interactive 3D visualization. Again, the 3D coordinates are only a visualization of the original vectors; they are not the embeddings used by Chroma for retrieval.

In [19]:
# Let's try 3D!

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

# Complete Mental Model

```text
Documents
    ↓
Load Documents
    ↓
Chunk Documents
    ↓
Embedding Model
    ↓
One Vector per Chunk
    ↓
Chroma Vector Database
```

Later, the retrieval side of RAG will look like:

```text
User Query
    ↓
Embedding Model
    ↓
Query Vector
    ↓
Similarity Search
    ↓
Top-K Relevant Chunks
    ↓
LLM
    ↓
Answer
```

The important distinction is that the embedding model converts text into vectors, while the generative LLM generates text. This notebook mainly covers the indexing side of RAG.